# 11. Functional Programming (5+ Years Interview Guide)
Mastering pure functions, lambda closures, map/filter/reduce pipelines, late-binding variable capture, partial application, and itertools.

### Key 5-Year Interview Concepts Covered:
- **First-Class & Pure Functions**: Deterministic execution, side-effect elimination, and immutability.
- **Lambda Closures & Late-Binding Gotcha**: Why `[lambda: i for i in range(5)]` returns 4 for all lambdas and how to fix it.
- **Functional Pipelines**: `map()`, `filter()`, `functools.reduce()`, and comparing them with comprehensions.
- **Partial Application & Composition**: Pre-binding arguments with `functools.partial` and stream processing with `itertools`.

This notebook uses the shared Fintech dataset `data/raw_transactions.csv` for interview scenario problems at the end.

In [1]:
# Setup: Locate the Shared Dataset
import os
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
print("Using CSV file path:", csv_path)

Using CSV file path: data/raw_transactions.csv


### 1. Anonymous Functions (`lambda`)
**Explanation**: Lambda expressions create small, anonymous function objects inline without the `def` statement. Lambdas are syntactically restricted to a single expression whose evaluated value is implicitly returned. They are best used as short throwaway callbacks for sorting keys or higher-order functions.

**Syntax**: `lambda x, y: x + y`

In [2]:
multiplier_lambda = lambda value: value * 2
print(multiplier_lambda(5))

10


### 2. Custom Sorting Keys with Lambdas
**Explanation**: The `key=` parameter in `sorted()` and `list.sort()` accepts a single-argument function mapping each item to a sortable comparison key. Using `key=lambda x: x['amount']` extracts comparison attributes cleanly in O(N) key calculations rather than calling custom comparisons on every element comparison.

**Syntax**: `sorted(records, key=lambda r: r['amount'], reverse=True)`

In [3]:
pairs_list = [(1, 'b'), (2, 'a')]
pairs_list.sort(key=lambda item: item[1])
print(pairs_list)

[(2, 'a'), (1, 'b')]


### 3. Late Binding Trap in Loop Closures
**Explanation**: Python closures bind free variables by name (reference), not by value at creation time. In `funcs = [lambda: i for i in range(5)]`, all 5 lambdas reference the exact same variable `i` in the enclosing scope. When invoked after the loop, `i` has its final value `4`, so all functions return `4`!

**Syntax**: `[lambda: i for i in range(5)]  # Gotcha: all return final i`

In [4]:
trap_list = [lambda: loop_index for loop_index in [1, 2]]
print([lambda_func() for lambda_func in trap_list])

[2, 2]


### 4. Fixing Late Binding with Default Parameters
**Explanation**: To capture the current value of a loop variable at closure creation time, pass it as a default parameter: `lambda val=i: val`. Because default arguments are evaluated when the function is defined, each lambda captures its own frozen copy of `i` in its `__defaults__` tuple.

**Syntax**: `[lambda val=i: val for i in range(5)]  # Fixed: returns 0, 1, 2, 3, 4`

In [5]:
safe_list = [lambda parameter_name=loop_index: parameter_name for loop_index in [1, 2]]
print([lambda_func() for lambda_func in safe_list])

[1, 2]


### 5. Mapping Iterables (`map()`)
**Explanation**: `map(function, iterable)` applies a function to every item in an iterable, returning a memory-efficient lazy iterator in O(1) initial time. While list comprehensions are often preferred for readability in simple transformations, `map()` is faster when applying existing named C-functions (e.g. `map(int, list_of_strings)`).

**Syntax**: `map(transform_fn, iterable)`

In [6]:
payout_records = [1, 2]
print(list(map(lambda x: x**2, payout_records)))

[1, 4]


### 6. Filtering Iterables (`filter()`)
**Explanation**: `filter(predicate, iterable)` yields only elements for which `predicate(item)` evaluates to `True`. Passing `None` as the predicate (`filter(None, items)`) filters out all falsy elements (removing empty strings, zeros, and `None` values efficiently).

**Syntax**: `filter(is_valid, iterable)` / `filter(None, sequence)`

In [7]:
payout_records = [1, 2, 3]
print(list(filter(lambda x: x%2==0, payout_records)))

[2]


### 7. Cumulative Reductions (`functools.reduce()`)
**Explanation**: `functools.reduce(function, iterable, [initializer])` applies a two-argument function cumulatively to sequence items to reduce the sequence to a single aggregate value. It is the Python equivalent of `foldl` in functional languages.

**Syntax**: `from functools import reduce; total = reduce(lambda acc, x: acc + x, numbers, 0)`

In [8]:
from functools import reduce
payout_records = [1, 2, 3]
print(reduce(lambda acc, val: acc+val, payout_records))

6


### 8. Functional Pipelines vs Comprehensions
**Explanation**: Chaining `map()` and `filter()` creates composable processing pipelines without allocating intermediate lists. In Python, list/generator comprehensions `(f(x) for x in seq if p(x))` are generally preferred for readability unless using pre-existing library callables.

**Syntax**: `(transform(x) for x in stream if is_valid(x))`

In [9]:
payout_records = [1, 2, 3]
print(list(map(lambda x: x*2, filter(lambda x: x>2, payout_records))))

[6]


### 9. Lazy Evaluation & Iterator Chaining
**Explanation**: Functional iterators (`map`, `filter`, generator expressions) compute elements on-demand as requested by consumers. This enables streaming and processing multi-gigabyte files with constant O(1) memory footprint.

**Syntax**: `lazy_stream = (process(line) for line in file_handle)`

In [10]:
lazy_iterator = map(lambda x: x, [1, 2])
print(type(lazy_iterator))

<class 'map'>


### 10. Boolean Reductions (`any()` & `all()`)
**Explanation**: `any(iterable)` checks if at least one item is truthy (short-circuiting immediately upon finding one), while `all(iterable)` verifies all items are truthy (short-circuiting on the first falsy item). Both accept lazy generators to avoid redundant computation.

**Syntax**: `has_fraud = any(tx.is_fraudulent for tx in transactions)`

In [11]:
print('Any True?:', any([False, True]))

Any True?: True


### 11. High-Performance Function Operators (`operator` module)
**Explanation**: The `operator` module exports standard Python operators as C-level functions: `operator.add`, `operator.mul`, `operator.itemgetter('amount')`, `operator.attrgetter('id')`. Using `itemgetter` instead of lambdas runs at native C speed and is the industry standard for high-performance sorting and grouping.

**Syntax**: `import operator; key_fn = operator.itemgetter('amount')`

In [12]:
from operator import itemgetter
pairs_list = [(1, 'b'), (2, 'a')]
pairs_list.sort(key=itemgetter(1))
print(pairs_list)

[(2, 'a'), (1, 'b')]


### 12. Partial Function Application (`functools.partial`)
**Explanation**: `functools.partial(func, *args, **kwargs)` freezes a subset of arguments and keywords of a function, returning a new callable with a simpler signature. It is heavily used in event handlers, callback pipelines, and multithreading pool mappings.

**Syntax**: `from functools import partial; usd_to_eur = partial(convert_currency, rate=0.92)`

In [13]:
from functools import partial
def add_values(a, b): return a + b
add_five = partial(add_values, 5)
print(add_five(10))

15


### 13. Running Accumulations (`itertools.accumulate`)
**Explanation**: `itertools.accumulate(iterable, [func])` yields running cumulative totals (or custom binary operations). Unlike `reduce()` which returns only the final scalar, `accumulate` yields every intermediate running state lazily.

**Syntax**: `from itertools import accumulate; running_totals = list(accumulate(amounts))`

In [14]:
from itertools import accumulate
print(list(accumulate([1, 2, 3])))

[1, 3, 6]


### 14. Chaining Iterables (`itertools.chain`)
**Explanation**: `itertools.chain(*iterables)` concatenates multiple sequences lazily without creating a combined list in memory. `itertools.chain.from_iterable(nested_iterables)` flattens an iterable of sequences in O(1) auxiliary space.

**Syntax**: `from itertools import chain; combined = chain(list_a, list_b, list_c)`

In [15]:
from itertools import chain
print(list(chain([1], [2])))

[1, 2]


### 15. Grouping Sequences (`itertools.groupby`)
**Explanation**: `itertools.groupby(iterable, key=None)` groups consecutive elements that share the same key. CRITICAL GOTCHA: The input sequence MUST be sorted by the key function beforehand; otherwise, `groupby` creates new groups whenever the key value changes.

**Syntax**: `from itertools import groupby; groups = groupby(sorted_items, key=key_fn)`

In [16]:
from itertools import groupby
for group_key, group_generator in groupby([('a', 1), ('a', 2)], lambda x: x[0]):
    print(group_key, list(group_generator))

a [('a', 1), ('a', 2)]


## Section 3: Fintech Senior Interview Scenarios
**Explanation**: Stream processing, partial currency conversion binding, and running transaction balance reconciliation.


In [17]:
# Solution:
from functools import reduce
rows = []
with open(csv_path, 'r') as f:
    f.readline()
    for _ in range(50):
        rows.append(f.readline().strip().split(','))
        
failed_txs = filter(lambda r: r[5] == 'Failed', rows)
amounts = map(lambda r: float(r[3]) if r[3] not in ('', 'NaN') else 0.0, failed_txs)
failed_sum = reduce(lambda acc, val: acc + val, amounts, 0.0)
print('Total Failed Sum:', failed_sum)


Total Failed Sum: 10656.119999999999


### Q2: Currency Conversion with `functools.partial`
**Explanation**: **Scenario**: Construct a partial function binding the EUR/USD conversion rate (0.92) to map transaction amounts across the dataset efficiently.

**Syntax**: `eur_converter = partial(convert_currency, rate=0.92)`

In [18]:
# Solution:
from functools import partial
def convert(val, rate): return round(val * rate, 2)
usd_to_eur = partial(convert, rate=0.92)

with open(csv_path, 'r') as f:
    f.readline()
    for _ in range(5):
        row = f.readline().strip().split(',')
        amt = float(row[3]) if row[3] not in ('', 'NaN') else 0.0
        print('USD:', amt, '-> EUR:', usd_to_eur(amt))


USD: 1216.33 -> EUR: 1119.02
USD: 324.99 -> EUR: 298.99
USD: 136.66 -> EUR: 125.73
USD: 124.21 -> EUR: 114.27
USD: 1284.68 -> EUR: 1181.91
